# WAXAL ASR - Zero-Shot Whisper Submission

Runs **Whisper Small** zero-shot on test audio (Luganda, Lingala, Shona)
and generates a submission CSV for the Zindi competition.

No fine-tuning needed. Downloads only test splits (~1.3 GB total).

**Steps:** Install deps → Setup → Download test data → Load model → Generate submission

## 1. Install Dependencies

In [1]:
!pip install -q soundfile "datasets==3.2.0" transformers
print("Dependencies installed.")

Dependencies installed.


## 2. Setup & Imports

In [2]:
import os, csv
from pathlib import Path

import datasets
import numpy as np
import torch
import transformers
from tqdm.auto import tqdm

PROJECT_ROOT = Path(".").resolve().parent

# Load HF token from .env (gitignored)
env_file = PROJECT_ROOT / ".env"
if env_file.exists():
    for line in env_file.read_text().strip().splitlines():
        if "=" in line and not line.startswith("#"):
            k, v = line.split("=", 1)
            os.environ[k.strip()] = v.strip()
    print("HF_TOKEN loaded from .env")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
print(f"datasets version: {datasets.__version__}")
print(f"transformers version: {transformers.__version__}")

c:\Users\ADMIN\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


HF_TOKEN loaded from .env
Device: cuda
datasets version: 3.2.0
transformers version: 5.13.1


## 3. Load Test Data from Local Parquet Files

Loads the 3 test parquet files from the `data/` folder.

In [3]:
LANGUAGES = ["lug", "lin", "sna"]
SAMPLE_RATE = 16_000
DATA_DIR = PROJECT_ROOT / "data"

print("Loading test data from local parquet files...\n")

test_data = {}
for lang in LANGUAGES:
    parquet_path = DATA_DIR / f"{lang}-test-00000.parquet"
    print(f"  {parquet_path.name} ...", end=" ", flush=True)
    ds = datasets.Dataset.from_parquet(str(parquet_path))
    ds = ds.cast_column("audio", datasets.Audio(sampling_rate=SAMPLE_RATE))
    test_data[lang] = ds
    print(f"OK ({len(ds)} examples)")

print("\nAll test data loaded.")

Loading test data from local parquet files...

  lug-test-00000.parquet ... 

Generating train split: 638 examples [00:02, 309.46 examples/s]


OK (638 examples)
  lin-test-00000.parquet ... 

Generating train split: 1832 examples [00:01, 1018.23 examples/s]


OK (1832 examples)
  sna-test-00000.parquet ... 

Generating train split: 1596 examples [00:02, 567.74 examples/s]


OK (1596 examples)

All test data loaded.


## 4. Load Whisper Small

In [4]:
MODEL_ID = "openai/whisper-small"

processor = transformers.WhisperProcessor.from_pretrained(MODEL_ID)
model = transformers.WhisperForConditionalGeneration.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16
)

# Disable forced decoder IDs so the model can transcribe any language
model.config.forced_decoder_ids = None
model.config.suppress_tokens = []
model.generation_config.forced_decoder_ids = None
model.generation_config.suppress_tokens = []

model = model.to(device)
model.eval()

params = sum(p.numel() for p in model.parameters()) / 1e6
print(f"Model: {MODEL_ID} ({params:.1f}M params)")
print(f"Loaded on {device}")
if torch.cuda.is_available():
    print(f"GPU memory: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

Loading weights: 100%|██████████| 479/479 [00:00<00:00, 769.18it/s]


Model: openai/whisper-small (241.7M params)
Loaded on cuda
GPU memory: 0.50 GB


## 5. Generate Submission CSV

Transcribes all ~4,253 test samples using Whisper zero-shot. Progress bar per language.

In [ ]:
test_csv_path = PROJECT_ROOT / "Test.csv"
sample_csv_path = PROJECT_ROOT / "SampleSubmission.csv"
submission_dir = PROJECT_ROOT / "submissions"
submission_dir.mkdir(parents=True, exist_ok=True)
submission_path = submission_dir / "submission_zero_shot.csv"

# Read test IDs and group by language
test_ids = []
with open(test_csv_path, "r", encoding="utf-8") as fh:
    for row in csv.DictReader(fh):
        test_ids.append(row["ID"])

# Group IDs by language prefix (e.g. "lug_96114" -> "lug")
lang_to_ids = {}
for tid in test_ids:
    lang = tid.split("_")[0]
    lang_to_ids.setdefault(lang, []).append(tid)

print(f"Test set: {len(test_ids)} samples")
for lang, ids in lang_to_ids.items():
    print(f"  {lang}: {len(ids)} samples")

# Run zero-shot inference
predictions = {}

for lang in LANGUAGES:
    if lang not in lang_to_ids:
        print(f"\nSkipping {lang} (no test IDs)")
        continue

    ds = test_data[lang]
    needed_ids = set(lang_to_ids[lang])

    # Build ID lookup: dataset index -> full test ID
    id_lookup = {}
    for idx in range(len(ds)):
        raw_id = str(ds[idx]["id"])
        full_id = f"{lang}_{raw_id}" if not raw_id.startswith(lang) else raw_id
        if full_id in needed_ids:
            id_lookup[idx] = full_id

    print(f"\n{lang}: transcribing {len(id_lookup)} samples...")

    for idx in tqdm(sorted(id_lookup.keys()), desc=f"Predict {lang}"):
        example = ds[idx]
        audio_array = np.asarray(example["audio"]["array"], dtype=np.float32)

        input_features = processor.feature_extractor(
            audio_array,
            sampling_rate=SAMPLE_RATE,
            return_tensors="pt",
        ).input_features.to(device=device, dtype=torch.float16)

        with torch.no_grad():
            pred_ids = model.generate(input_features, max_new_tokens=225)

        transcript = processor.tokenizer.decode(
            pred_ids[0], skip_special_tokens=True
        ).strip()
        predictions[id_lookup[idx]] = transcript

# Write submission CSV
with open(submission_path, "w", encoding="utf-8", newline="") as fh:
    writer = csv.writer(fh)
    writer.writerow(["ID", "Target"])
    for tid in test_ids:
        writer.writerow([tid, predictions.get(tid, "")])

print(f"\nSubmission written to: {submission_path}")
print(f"Predictions: {len(predictions)} / {len(test_ids)}")

# Validate against SampleSubmission.csv
if sample_csv_path.exists():
    with open(sample_csv_path, "r", encoding="utf-8") as fh:
        expected_ids = {row["ID"] for row in csv.DictReader(fh)}
    with open(submission_path, "r", encoding="utf-8") as fh:
        submitted_ids = {row["ID"] for row in csv.DictReader(fh)}
    missing = expected_ids - submitted_ids
    empty = sum(1 for tid in test_ids if not predictions.get(tid, ""))
    if missing:
        print(f"WARNING: Missing {len(missing)} IDs!")
    elif empty:
        print(f"WARNING: {empty} IDs have empty transcriptions")
    else:
        print("Submission validation PASSED - all IDs present with transcriptions")

Test set: 4253 samples
  lug: 638 samples
  lin: 1866 samples
  sna: 1749 samples

lug: transcribing 638 samples...


Predict lug:   0%|          | 0/638 [00:00<?, ?it/s][transformers] Both `max_new_tokens` (=225) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created i